# **Logistic Regression**  

**Author**: David Thébault.  
**Date**: October 18, 2025.  
**Objective**: Understand, implement, and apply logistic regression for diabetes prediction.

## **Table of Contents**
1. [Introduction to Logistic Regression](#introduction-to-logistic-regression)
2. [Theory and Fundamentals](#theory-and-fundamentals)
   - Model and Assumptions
   - Coefficient Interpretation
   - Cost Function and Optimization
3. [Implementation with `statsmodels`](#implementation-with-statsmodels)
   - Data Preparation
   - Training and Interpretation
4. [Implementation "From Scratch"](#implementation-from-scratch)
   - Gradient Descent Algorithm
   - Testing and Validation
5. [Application: Diabetes Prediction](#application-diabetes-prediction)
   - Data Preparation
   - Prediction and Interpretation
6. [Appendices](#appendices)
   - Mathematical Demonstrations
   - Wald Statistic and Deviance
7. [References](#references)
---

## Introduction to Logistic Regression {#introduction-to-logistic-regression}

The Logistic regression applies to cases where:

* $Y$ is a random qualitative variable with 2 categories (a binary variable by convention, $Y = 0$ if the event does not occur, and $Y = 1$ if it does),
* $X_1,\ldots,X_k$ are non-random qualitative or quantitative variables ($K$ explanatory variables in total).

* $(Y, X_1,\ldots,X_k)$ represent the population variables, from which a sample of $n$ individuals $(i)$ is drawn, and $(y, x_i)$ is the vector of observed realizations of $(Y_i, X_i)$ for each individual in the sample.

Unlike simple linear regression, logistic regression estimates **the probability** of an event occurring, rather than predicting a specific numerical value.

## Theory and Fundamentals {#theory-and-fundamentals}

### Probabilistic Model

This section describes the underlying probabilistic framework of the model, focusing on the random variable and its distribution.

**The Model**  

The variable $Y_i$ follows a **Bernoulli distribution** with parameter $p_i$ representing the probability that $Y_i=1$.    

$$Y_i \sim B(p_i)$$

The probabilities are defined as:  

$$P(Y_i=1) = p_i \quad, \quad P(Y_i = 0) = 1 - p_i$$

This can also be written as:: 

$$P(Y_i = k) = {p_i}^k(1 - p_i)^{1-k} \quad \text{for k} \in \{0, 1\}$$

### LOGIT Model

This section focuses on the specific form of the logistic regression model, which is a type of probabilistic model.  

**The Linear Logit Model**

To ensure that the expected value of $Y, E(Y)$, only takes values between 0 and 1, we use the **logistic function**:  

$$f(x) = \dfrac{\text{exp(x)}}{1 + \text{exp(x)}} = p$$

or equivalently:  

$$f(x) = \dfrac{1}{1 + \text{exp(-x)}} = p$$

This guarantees that $0 < f(x) < 1$, so $E[Y]$ can represent a valid probability.  

The **logit function** transforms a probability $p$ into an **unrestricted real value**:

**Notations**:
* $X = (1,X_1, \ldots, X_k)$ vector of predictors, including the intercept term.
* $\beta = (\beta_0,\beta_1, \ldots, \beta_k)$ vector of coefficients.

**Logit Transformation**

The logit of a probability $p$ is defined as:

$$\text{logit}(p) = \text{log}(\dfrac{p}{1 - p})$$

In the context of logistic regression, the logit of the probability is modeled as a linear combination of the predictors:

$$\text{logit}(p) = \beta .X$$

Explicitly, this can be written as:

$$\text{logit}(p) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \ldots + \beta_k X_k$$

Or equivalently:

$$\text{log}\left( \dfrac{p}{1-p} \right) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \ldots + \beta_k X_k$$

**Inverse Transformation: Logistic Function**. 

To convert the linear predictor back into a probability, we use **the logistic function (the inverse of the logit function)**:

$$p = \frac{1}{1 + \exp(-\beta .X)}$$

Demonstration: 

$$p(x) = \dfrac{1}{1 + \exp(-\beta x)}$$

$$\underset{inverse}   \iff \dfrac{1}{p} = 1 + \exp(-\beta x)$$

$$\iff \dfrac{1}{p} - 1 = \exp(-\beta x)$$

$$\iff \dfrac{1}{p} - \dfrac{p}{p} = \exp(-\beta x)$$

$$\iff \dfrac{1-p}{p} = \exp(-\beta x)$$

$$\iff \log(\dfrac{1-p}{p}) = -\beta x$$

$$\iff \log(\dfrac{p}{1-p}) = \beta x$$

For simplicity, we often write $p$ instead of $p(x)$ when the context is clear. 

### Model Assumptions {#model-assumptions}

For the **logistic regression model** to be valid and generalizable, the following key assumptions must be satisfied. Violations of these assumptions can lead to biased or inefficient estimates, and may compromise the interpretability of the model.

---

#### 1. Linearity of Log-Odds {#linearity-of-log-odds}
**Assumption:**
The relationship between **each continuous predictor** ($X_j$) and the **log-odds** of the outcome ($Y=1$) must be **linear**. Mathematically, this means:
$$\log\left(\frac{P(Y=1|X)}{1 - P(Y=1|X)}\right) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_k X_k$$

**Implications:**
- If the true relationship is non-linear (e.g., quadratic or U-shaped), the linear assumption will not hold.
- **How to check:** Plot the log-odds against each continuous predictor. If the relationship appears curved, consider adding polynomial terms or splines.

**Example:**
If `age` is a predictor, the log-odds of diabetes should increase or decrease linearly with `age`. If the relationship is non-linear, you might need to add a quadratic term (e.g., `age²`).

---

#### 2. No Multicollinearity {#no-multicollinearity}
**Assumption:**
Predictor variables should **not be highly correlated** with each other. High multicollinearity can inflate the variance of coefficient estimates, making them unstable and difficult to interpret.

**Implications:**
- Multicollinearity does not bias the model, but it reduces the precision of the estimated coefficients.
- **How to check:** Use **Variance Inflation Factor (VIF)**. A VIF > 5 or 10 indicates problematic multicollinearity.

**Example:**
If your model includes both `blood_pressure` and `blood_pressure_squared`, these variables are likely to be highly correlated. Consider removing one or using regularization (e.g., Lasso or Ridge).

---

#### 3. Additivity {#additivity}
**Assumption:**
The effect of each predictor on the log-odds of the outcome is **additive**. This means the contribution of each predictor to the log-odds is independent of the values of the other predictors.

**Implications:**
- If there are **interaction effects** (e.g., the effect of `age` on diabetes depends on `BMI`), they must be explicitly modeled.
- **How to check:** Test for interactions by adding product terms (e.g., `age × BMI`) to the model and evaluating their significance.

**Example:**
If the effect of `glucose` on diabetes risk depends on `age`, you should include an interaction term like `glucose × age` in your model.

---

#### 4. Independence of Observations {#independence-of-observations}
**Assumption:**
The model assumes that **observations are independent** of each other. This is critical for the validity of standard errors and hypothesis tests.

**Implications:**
- If observations are correlated (e.g., repeated measurements from the same individual or clustered data), standard errors will be underestimated, leading to inflated Type I error rates.
- **How to check:** Look for patterns in residuals or use tests for autocorrelation (e.g., Durbin-Watson test for time-series data).

**Example:**
If your dataset includes multiple measurements from the same patient over time, you should use a **mixed-effects logistic regression** or **GEE (Generalized Estimating Equations)** to account for within-patient correlation.

---

#### Summary Table of Assumptions
| Assumption                | Description                                                                                     | How to Check                                  | Solution if Violated                          |
|---------------------------|-------------------------------------------------------------------------------------------------|-----------------------------------------------|-----------------------------------------------|
| **No Multicollinearity**  | Predictors are not highly correlated.                                                         | Calculate VIF.                                | Remove predictors or use regularization.     |
| **Linearity of Log-Odds**  | Continuous predictors have a linear relationship with the log-odds of the outcome.             | Plot log-odds vs. predictors.                 | Add polynomial/spline terms.                 |
| **Additivity**            | The effect of each predictor is independent of other predictors.                              | Test for interactions.                       | Add interaction terms.                        |
| **Independence**          | Observations are independent (no clustering or repeated measures).                           | Check residuals for patterns.                | Use mixed-effects models or GEE.              |

---



## Practical Implementation of Logistic Regression

#### Recommendations
1. **Always check assumptions** before interpreting your model.
2. **Use diagnostic plots** (e.g., residual plots, VIF) to identify violations.
3. **Consider regularization** (Lasso/Ridge) if multicollinearity is an issue.
4. **Model interactions** if additivity is violated.
5. **Use robust standard errors** or mixed-effects models if observations are not independent.


### **Checking the first model assumption: No Multicolinearity**

**1. $\text{VIF}$ for numeric variables** 

VIF (Variance Inflation Factor) is the most common method to detect multicollinearity and works well for continuous variables.  
Fit a linear regression model for each predictor against all other predictors and calculate VIF as $R^2$ inverse.  

Interpretation:

* $\text{VIF} < 5$: No problematic multicollinearity.
* $5 \leq \text{VIF} < 10$: Moderate multicollinearity, monitor closely.
* $\text{VIF} \geq 10$: High multicollinearity, needs correction.  

**1'. $\text{GVIF}$ for categorical variables (dummy variables)** 

For categorical variables, use Generalized Variance Inflation Factor ($\text{GVIF}$) instead of $\text{VIF}$.  

$\text{GVIF}$ provides a more accurate measure of multicollinearity for categorical variables because it takes into account the degrees of freedom $(\text{d})$ associated with categorical variables.

Interpretation:
* $\text{GVIF} = \text{VIF}^{\frac{1}{2\text{d}}} < 2$: No problematic multicollinearity.
* $\text{GVIF} = \text{VIF}^{\frac{1}{2\text{d}}} \geq 2$: Indicates potential multicollinearity.  

**Degrees of freedom $(d)$** refer to the number of independent pieces of information available to estimate a parameter.   
For a categorical variable with $k$ levels, the degrees of freedom are $k−1$ because one level is used as the reference category.  

When we create dummy variables from a categorical variable, the dummy variables are not independent of each other.  
To avoid ta perfect multicollinearity, we drop one dummy variable (reference category), leaving you with $k−1$ dummy variables.  

**VIF** treats each dummy variable as an independent predictor, which can overestimate multicollinearity because it doesn't account for the shared variance among dummy variables from the same categorical variable.  
**GVIF** adjusts for this by considering the degrees of freedom of the categorical variable. It essentially "penalizes" the VIF value to account for the fact that the dummy variables are related.  

**VIF** overestimates multicollinearity for **dummy variables** because it treats them as independent predictors.

**3. Pearson correlation for numeric variables** 

Use Pearson correlation to detect multicolinearity amoung numeric variables.  

Interpretation:
* Values close to 1 or -1 indicate strong linear relationships.

**4. Point-Biserial correlation**

Measure the association (pairwise correlation) between numeric and dummy variables. It is essentially a special case of Pearson correlation where one variable is binary.

Interpretation:

* High absolute values (close to 1 or -1) indicate a strong association between the numeric and dummy variables.

**5. Eigenvalues and Condition Number**  

Check the eigenvalues of the correlation matrix of numeric and dummy variables to detect multicollinearity.

This analysis applies to the entire set of predictors (both numeric and dummy variables) included in the model.
Ensure all variables are numeric (dummy variables are already numeric, encoded as 0 or 1).

Interpretation:

Condition Number: 
* A high condition number (e.g., > 30) indicates potential multicollinearity. 

Eigenvalues: 
* A scree plot with some eigenvalues close to zero suggests multicollinearity.
* If one or more eigenvalues are near zero, it indicates that some predictors are linear combinations of others.

**6. Pair Plot for Dummy Variables**  

Visualize the relationships between dummy variables using a pair plot.   

Interpretation:

Look for patterns that indicate strong associations between dummy variables.

A $\text{VIF}$ of 1.36 for glucose means that the variance of the regression coefficient for glucose is multiplied by 1.36 due to multicollinearity with all other variables (both continuous and dummy) in the model.  
This indicates a low level of multicollinearity overall.
Given a $\text{VIF}$ of 1.36 for glucose, we can work backward to find $R^2_j$:
$$\text{VIF}_j = \dfrac{1}{1 - R^2_j}=1.36$$

Solving for $R^2_j$:
$$R^2_j \approx 0.2647$$

* This means that about 26.47% of the variance in the glucose variable is explained by all the other predictor variables in the model.
* This is a relatively low percentage, indicating that glucose is not highly collinear with the other predictors.
$\text{VIF} = 1.36$:

The variance of the regression coefficient for glucose is inflated by a factor of 1.36 due to its relationship with all other predictors.
This is considered a low $\text{VIF}$ value, indicating that multicollinearity is not a significant issue for glucose in this model.

**Definition of $\text{VIF}$**

The Variance Inflation Factor $(\text{VIF})$ measures the increase in the variance of regression coefficients due to multicollinearity with all other variables in the model.  
It does not specifically measure multicollinearity with a single variable but rather the cumulative effect of all other variables on a given variable.

**Mathematical formula of $\text{VIF}$**

$\text{VIF}$ for a predictor variable $X_j$​ in a regression model is defined as:  

$$\text{VIF}_j = \dfrac{1}{1 - R^2_j}$$

Where $R^2_j$ is the coefficient of determination obtained by regressing the predictor $X_j$ on all the other predictor variables in the model. It ranges from 0 to 1.

* If $R^2_j$ is close to 0, it means $X_j$ is not well-explained by the other predictors, and $\text{VIF}$ will be close to 1, indicating no multicolinearity.
* If $R^2_j$ is close to 1, it means $X_j$ is well-explained by the other predictors, and $\text{VIF}$ will be large, indicating high multicolinearity.  

To calculate $R^2_j$, you regress $X_j$ on all other predictors in the model. Treat $X_j$ as the dependent variable. Treat all other predictors in the model as independent variables.

Fit a linear regression model to predict $X_j$​ using the other predictors.

$$X_j = \beta_0 + \beta_0 X_1 + \beta_2 X_2 + \ldots + \beta_{j-1}X_{j−1}+ \beta_{j+1} X{j+1} + \ldots + \beta_p X_p + \epsilon $$

**1. $\text{VIF}$ for continuous variables**

**2. Pearson correlation for numeric variables**

**4. Point-Biserial correlation**

**5. Eigenvalues and Condition Number**  

**6. Pair Plot for Dummy Variables**  

Visualize the relationships between dummy variables using a pair plot.   

Interpretation:

Look for patterns that indicate strong associations between dummy variables.

**7. V-Cramer for categorical variables**

**Cramer's V** is a measure of association between two nominal variables, useful for assessing multicollinearity among dummy variables.  
It ranges from 0 (no association) to 1 (perfect association).  

Interpretation:

**8. Pearson correlation for dummy variables**

#### **Fit the logistic regression model**

##### Logistic Regression with scipy

#### **Linearity of Log-Odds**

Predicted Probabilities and Log-Odds:

* The linearity of log-odds assumption in logistic regression refers to the relationship between continuous predictors and the log-odds of the outcome.
* To assess this, you need the predicted probabilities from a fitted logistic regression model. These probabilities are then transformed into log-odds.  

Visual Inspection:

* You plot the continuous predictors against the predicted log-odds to visually inspect if the relationship is linear.
* If the relationship is not linear, you may need to transform the predictor (e.g., using polynomials, splines, or categorization).

##### Calculate Predicted Probabilities and Log-Odds

##### Plot Log-Odds vs. Continuous Predictors

## Coefficients interpretation

**The Odds**

The odds are defined by:  

$$\text{Odds} = \dfrac{p}{1-p}$$


$\text{Where} \quad p = P(target=1|X)$

>_If a student has a 3 in 4 chance of passing and a 1 in 4 chance of failing, their odds are '3 to 1':_ $\text{Odds} = \dfrac{3/4}{1/4}=3$  

* **Notation:**
$$\text{Odds}(Y=1|X=0)=\dfrac{P(Y=1|X=0)}{1-P(Y=1|X=0)}$$

### **The Odds Ratio**

The odds ratio comparing the **probability of $target=1$** between individuals with value $X$ and those without it.

$$\text{Odds Ratio} = \dfrac{\text{Odds}(Y=1|X=1)}{\text{Odds}(Y=1|X=0)}$$

$$\text{Odds Ratio} = \dfrac{P(Y_i=1 | X=1)}{1 - P(Y_i=1 | X=1)} / \dfrac{P(Y_i=1 | X=0)}{1 - P(Y_i=1 | X=0)}$$

We know that logit is given by:

$$\text{logit}(p) = \text{log}(\dfrac{p}{1-p}) = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \ldots + \beta_k x_{ik}$$

### **Interpreting the Intercept**  

The intercept $\beta_0$​ represents the **log-odds of the outcome $Y=1$ when all predictors are equal to zero**.  
$\beta_0$​ defines the **baseline probability** of the outcome when all predictors are zero.  

⚠️ **Caveat**:  
This interpretation of $\beta_0$ is often not meaningful if some predictors cannot logically be zero (e.g., age=0, blood pressure). In such cases, $\beta_0$​ is primarily a mathematical component of the model and is rarely interpreted in isolation.  
  
  
* **Odds for the baseline group:**  

$$\text{Odds}(Y=1∣X_1=0)=\exp⁡(\beta_0)$$

* **Probability for the baseline group:**
$$P(Y=1∣X_1=0)=\dfrac{\exp⁡(\beta_0)}{1 + \exp(\beta_0)}$$


>If $X_1$​ is "smoking status" ($0$ = non-smoker, $1$ = smoker), then 
>* $\beta_0$​ gives the **log-odds** of the outcome for non-smokers  
>* $\exp(\beta_0)$ gives the **odds** of the outcome for non-smokers.  
>
>If $\beta_0 = -1$, then: $$\exp(\beta_0) = \exp(-1) \approx 0.37$$
>
>$$P(Y=1∣X_1=0)= \dfrac{0.37}{1 + 0.37} \approx 0.27$$
>
>27% of non-smokers are predicted to have the outcome (e.g., lung cancer), assuming no other predictors.  
>It is the observed proportion of lung cancer for non-smokers.  

### **Interpreting the Slope**

In a model with multiple predictors, each $\beta_i$​ (and its corresponding odds ratio $\exp(\beta_i)$ represents the effect of that predictor on the log-odds of $Y=1$, holding all other predictors constant.  
This is the key assumption of multivariable regression: ceteris paribus (all else being equal).

The coefficient $\beta_1$​ represents the **change in the log-odds of** $Y=1$ for a **one-unit change** in $X_1$​. The odds ratio $\exp(\beta_1)$​ quantifies how the odds of $Y=1$ change with $X_1$​.

**General Formula for Odds Ratio**

For any type of predictor $X_1$, the odds ratio for a one-unit increase is:  
$$\text{Odds Ratio} = \frac{\text{Odds}(Y=1 | X_1 = x+1)}{\text{Odds}(Y=1 | X_1 = x)} = \exp(\beta_1)$$

📌 **Note:**  
$\exp(\beta_1​)$ compares the odds of $Y=1$ between $X_1=1$ and $X_1=0$, controlling for all other variables in the model (all others features constant).

* Case: $X_1$ is Binary

For a binary predictor $X_1$​ (e.g., $0$ = non-smoker, $1$ = smoker), the odds ratio $\exp(\beta_1)$​ compares the odds of $Y=1$ between the two groups.

* **Logistic regression equation:**

$$\log\left(\dfrac{P(Y=1 | X_1)}{1 - P(Y=1 | X_1)}\right) = \beta_0 + \beta_1 1_{\{X_1 = 1\}}​$$

* **Odds ratio:**
$$\text{Odds Ratio} = \dfrac{P(Y=1 | X_1=1)}{1 - P(Y=1 | X_1=1)} / \dfrac{P(Y=1 | X_1=0)}{1 - P(Y=1 | X_1=0)} = \exp(\beta_1)$$

**Interpretation:**

* If $\exp(\beta_1) = 1$: No effect of the feature $X_1$​ on the odds of $Y=1$.
* If $\exp(\beta_1)>1$: The odds of $Y=1$ are higher when $X_1​=1$. The feature $X_1$​ is **positively associated** with the outcome.
* If $\exp(\beta_1) < 1$: The odds of $Y=1$ are lower when $X_1​=1$. The feature $X_1$​ is **negatively associated** with the outcome.

>**Example:**  
>If $\beta_1 = 0.7 \rightarrow \exp(\beta_1) \approx 2.01$. The odds of lung cancer for smokers $(X_1=1)$ are twice as high as for non-smokers $(X_1=0)$.

* Case: $X_1$ is Categorical

For a categorical predictor $X_1$ with more than two levels (e.g., color = red, green, blue), you use **dummy variables**. 

* **The logistic regression model becomes:**

$$\text{log}\left( \dfrac{P(Y=1)}{1 - P(Y=1)} \right) = \beta_0 + \beta_{green}1_{\{X_1 = \text{green}\}} + \beta_{blue}1_{\{X_1 = \text{blue}\}}$$

* **Reference Category ("red"):** When $1_{\{X_1 = \text{green}\}}=0$ and $1_{\{X_1 = \text{blue}\}}=0$, the log-odds are:

$$\text{log}\left( \dfrac{P(Y=1)}{1 - P(Y=1)} \right) = \beta_0$$

**Interpretation:** This means $\beta_0$​ represents the log-odds of $Y=1$ for the reference category ("red").

* **Category ("green"):** When $1_{\{X_1 = \text{green}\}}=1$ and $1_{\{X_1 = \text{blue}\}}=0$, the log-odds are:

$$\text{log}\left( \dfrac{P(Y=1)}{1 - P(Y=1)} \right) = \beta_0 + \beta_{green}$$

* **Category ("blue"):** When $1_{\{X_1 = \text{green}\}}=0$ and $1_{\{X_1 = \text{blue}\}}=1$, the log-odds are:

$$\text{log}\left( \dfrac{P(Y=1)}{1 - P(Y=1)} \right) = \beta_0 + \beta_{blue}$$

The **odds ratio for "blue" relative to the reference "red"** is:
$$\exp(\beta_{blue}) = \dfrac{\text{Odds}(Y=1 | \text{blue})}{\text{Odds}(Y=1 | \text{red})}$$

The same way, $\exp(\beta_{\text{green}})$​ compares the odds for "green" vs. the "red" reference.

>**Interpretation:**
>
>If $\exp(\beta_{\text{green}})​=1.5$, the odds of $Y=1$ are $1.5$ times higher for "green" compared to "red".

* ##### Case: $X_1$ is Quantitative

For a continuous predictor $X_1$ (e.g., age, blood pressure), the odds ratio $\exp(\beta_1)$​ represents the multiplicative change in the odds of $Y=1$ for a one-unit increase in $X_1$​.  

* **Logistic regression equation:**

$$\log\left(\dfrac{P(Y=1 | X_1)}{1 - P(Y=1 | X_1)}\right) = \beta_0 + \beta_1 X_1​$$

* **Odds ratio for a one-unit increase:**

$$\text{Odds Ratio} = \frac{\text{Odds}(Y=1 | X_1 = x+1)}{\text{Odds}(Y=1 | X_1 = x)} = \exp(\beta_1)​$$

**In short**: $\beta_1$​ captures the **constant log-odds** change per unit increase in $X_1$, so $\exp(\beta_1)$​ is the **odds ratio** for that one-unit change.

This holds regardless of the starting value of $X_1$​ because the model assumes a constant multiplicative effect on the odds (a key assumption of logistic regression).


>**Interpretation:**
>
>* If $\beta_1=0.095 \rightarrow \exp(\beta_1)=1.1$, the odds of $Y=1$ increase by 10% for each one-unit increase in $X_1$​.
>* If $X_1$​ is "years of smoking" and $\beta_1 = 0.7 \rightarrow  \exp(\beta_1) \approx 2.01$. For each additional year of smoking, the odds of lung cancer double.
>.

**Summary**

| Type of $X_1$         | Interpretation of $\exp(\beta_1)$                                       |
|-----------------------|-------------------------------------------------------------------------|
| Binary                | Compares odds of $Y=1$ between $X_1=1$ and $X_1=0$                      |
| Categorical           | Compares odds of $Y=1$ for a given category relative to the reference.  |
| Quantitative          | Multiplicative change in odds of $Y=1$ for a one-unit increase in $X_1$ |


6. Interpretation of the Plots

* Consistency of Log-Odds: The boxplot will show the distribution of predicted log-odds for each category. If the medians of the log-odds for each category are distinct, it suggests that the categorical predictor is having an effect on the log-odds of the outcome.

* Effect of Categories: The difference in log-odds between categories should reflect the coefficients in the logistic regression output.

**Interpreting the coefficients:**

For each feature, the exponentiated coefficient (exp(coef)) represents the change in odds for a one-unit increase in that feature, holding all other features constant.

For example, has the coefficient for 'pregnancies' is 0.1604, the odds ratio is exp(0.1604) ≈ 1.174037.  
This means that for each one-unit increase in pregnancies, the odds of the outcome occurring (e.g., having diabetes) increase by approximately 17.4%, assuming all other features remain constant.


**Diabetes Prediction: data preparation & model inference**

* Preparing new input data for prediction

* Using the model to make a prediction

- Transform continuous to categorical

## APPENDIX

### Interpreting the coefficients

**Demonstration:** The coefficient $\beta_1$​ represents the **change in the log-odds of** $Y=1$ for a **one-unit change** in $X_1$​ quantitative feature.  

Notations:  
$$\text{Odds}(Y=1|X=x+1)=P(Y=1|X=x+1) / (1-P(Y=1|X=x+1))$$

$$\text{Odds}(Y=1|X=x)=P(Y=1|X=x) / (1-P(Y=1|X=x))$$

We know that:  

$$\text{log(Odds)}(Y=1|X=x+1)=\beta_0 + \beta_1 \times (x+1)$$

$$\text{log(Odds)}(Y=1|X=x)=\beta_0 + \beta_1 \times x$$

By difference:

$$\text{log(Odds)}(Y=1|X=x+1) - \text{log(Odds)}(Y=1|X=x) =\beta_0 + \beta_1 \times (x+1) - (\beta_0 + \beta_1 \times x) = \beta_1$$

$$\text{log}\left(\dfrac{\text{Odds}(Y=1|X=x+1)}{\text{Odds}(Y=1|X=x)}\right) =\beta_1$$

**CQFD**

Note:  

$$\dfrac{\text{Odds}(Y=1|X=x+1)}{\text{Odds}(Y=1|X=x)} = \exp(\beta_1)$$

### Model formulation

The prediction $y_i=1$ of the logistic regression is defined:

$$\hat{y_i} = P(y_i=1 | x_i; \theta) = \frac{1}{1 + \exp(-\theta^Tx_i)} = h_{\theta}(x_i)$$

* If $y_i=1$, then $P(y_i|x_i; \theta)=P(y_i=1|x_i; \theta)$
* If $y_i=0$, then $P(y_i|x_i; \theta)=P(y_i=0|x_i; \theta) = 1 - P(y_i=1|x_i; \theta)$

We can write these two equations into a single one:  

$$P(y_i|x_i; \theta)=P(y_i=1|x_i; \theta)^{y_i}\times (1 - P(y_i=1|x_i; \theta))^{1-y_i}$$

With the notations:

$$P(y_i|x_i; \theta)=h_{\theta}(x_i)^{y_i}\times (1 - h_{\theta}(x_i))^{1-y_i}$$


### Likelihood function

The **likelihood** of the observations $y_i$ given the inputs $x_i$ and parameters $\theta$ is defined as:  

$$L(\theta) = \prod_{i=1}^n P(y_i|x_i;\theta) = \prod_{i=1}^n (h_{\theta}(x_i))^{y_i} (1 -h_{\theta}(x_i))^{1-y_i}$$

where the prediction of the logistic regression is defined:

$$h_{\theta}(x_i) = P(y_i=1 | x_i; \theta) = \frac{1}{1 + \exp(-\theta^Tx_i)}$$

The **log-likelihood** is defined as:  

$$l(\theta) = \log(L(\theta))=\sum_{i=1}^n[y_i \log (h_{\theta}(x_i)) + (1 - y_i) \log(1 - h_{\theta}(x_i))]$$

### Objective of Logistic Regession

The goal of learning a **logistic regression model** is to **minimize the cost function** by adjusting the parameters $\theta$.The cost function measures the average prediction error across all $n$ training samples.

### Cost function (general definition)

The cost function $J(\theta)$ is defined as the average penalty for prediction errors across the training set. Mathematically, it is expressed as:

$$J(θ) = -\frac{1}{n} \sum cost(h_\theta(x_i), y_i)$$

where:

* $h_\theta(x_i)$ the model's prediction for sample $x_i$
* $y_i$ the observed (true) value.

Alternatively, it can be written as:

$$J(θ) = \frac{1}{n} \sum cost(\hat{y_i}, y_i)$$

Where:

* $\hat{y_i} = h_{\theta}(x_i)$

### Cost function **log loss**

For logistic regression, the cost function is called **log loss** (or logistic loss).  
**Log loss** is derived from the log-likelihood $l(\theta)$ and is defined as:

$$J(\theta) = -\dfrac{1}{n}l(\theta) = -\frac{1}{n} \sum(y_i \log(h_\theta(x_i)) + (1 - y_i) \log(1 - h_\theta(x_i)))$$


#### How Log-Loss penalizes prediction errors

The log-loss function penalizes prediction errors based on the estimated probability from the model. It assigns higher penalties when predictions are far from the true labels—specifically:

- If the label $y_i = 1$ (positive class), the penalty is $-log(h_{\theta}(x_i))$. The closer $h_{\theta}(x_i)$ is to 0 (far from the true label), the higher the penalty ($-\log(0^+) \approx + \infty$).  

- If the label $y_i = 0$ (negative class), the penalty is $-log(1-h_{\theta}(x_i))$, the closer $h_{\theta}(x_i)$ is to 1 (far from the true label), the higher the penalty ($-\log(1 -1^+) \approx + \infty$).  

The two cases are combined into a single formula for observation $i$:
$$y_i log(h_\theta(x_i)) + (1 - y_i) log(1 - h_\theta(x_i))$$


**Key insights:**

* **Log loss** evaluates how well the model fits the training data.
* The log loss is the negative average the log likelihood.
* **Higher likelihood** leads to **lower log loss** (since $J(\theta) = -\frac{1}{n}l(\theta)$).
* The log-loss function heavily penalizes confident wrong predictions.

### Optimizing the parameters

By minimizing the cost function $J(\theta)$, we aim to find the parameters $\theta$ that maximize the likelihood of observing the training data given the model parameters.  

To achieve this, we use an iterative optimization method, **gradient descent**, to find the values of $\theta$ that minimize the cost function over the training set:

$$\underset{\theta}{minimize}(J(\theta))$$

### Parameter estimation methods

#### Gradient Descent

The gradient descent only uses the **gradient** (first derivative) of the cost function to update the parameters $\theta$ in the opposite direction of the gradient, scaled by a learning rate $\alpha$.  

To compute the gradient $\nabla J(\theta)$ we start by transforming the expression of:

$$\log(h_\theta(x_i)) = \log\left(\frac{1}{1 + \exp(-\theta^Tx_i)}\right) = -\log(1 + \exp(-\theta^Tx_i))$$

And:  

$$\log(1 - h_\theta(x_i)) = \log\left(1 - \frac{1}{1 + \exp(-\theta^Tx_i)}\right)$$

$$\log(1 - h_\theta(x_i)) = \log\left(\frac{1 + \exp(-\theta^Tx_i) - 1}{1 + \exp(-\theta^Tx_i)}\right)$$

$$\log(1 - h_\theta(x_i)) = \log\left(\frac{\exp(-\theta^Tx_i)}{1 + \exp(-\theta^Tx_i)}\right)$$

$$\log(1 - h_\theta(x_i)) = \log(\exp(-\theta^Tx_i)) - \log({1 + \exp(-\theta^Tx_i)})$$

$$\log(1 - h_\theta(x_i)) = -\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)})$$

We integrate these modifications:  

$$J(\theta) = -\frac{1}{n} \sum[y_i (-\log(1 + \exp(-\theta^Tx_i))) + (1 - y_i) (-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}))]$$

$$J(\theta) = -\frac{1}{n} \sum[y_i (-\log(1 + \exp(-\theta^Tx_i))) + (1 - y_i) (-\theta^Tx_i - \log(1 + exp(-\theta^Tx_i)))]$$

$$J(\theta) = -\frac{1}{n} \sum[y_i (-\log(1 + \exp(-\theta^Tx_i))) -\theta^Tx_i - \log(1 + \exp(-\theta^Tx_i)) + y_i \theta^Tx_i  + y_i \log(1 + \exp(-\theta^Tx_i))]$$

$$J(\theta) = -\frac{1}{n} \sum[- y_i \log(1 + \exp(-\theta^Tx_i)) -\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) + y_i \theta^Tx_i  + y_i \log(1 + \exp(-\theta^Tx_i))]$$

$$J(\theta) = -\frac{1}{n} \sum[\cancel{- y_i \log(1 + \exp(-\theta^Tx_i))} -\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) + y_i \theta^Tx_i  + \cancel{y_i \log(1 + \exp(-\theta^Tx_i))}]$$

$$J(\theta) = -\frac{1}{n} \sum[-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) + y_i \theta^Tx_i  ]$$

$$J(\theta) = -\frac{1}{n} \sum[y_i \theta^Tx_i  -\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) ]$$

with:

$$-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) = - \log(\exp(\theta^T x_i)) - \log(1 + \exp(-\theta^Tx_i))$$

$$-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) = -(\log(\exp(\theta^T x_i)) + \log(1 + \exp(-\theta^Tx_i)))$$

$$-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) = -\log[\exp(\theta^T x_i)(1 + \exp(-\theta^Tx_i))]$$

$$-\theta^Tx_i - \log({1 + \exp(-\theta^Tx_i)}) = -\log(\exp(\theta^T x_i) + 1)$$

$$J(\theta) = -\frac{1}{n} \sum[y_i \theta^Tx_i  -\log(\exp(\theta^T x_i + 1)) ]$$

$$J(\theta) = -\frac{1}{n} \sum[y_i \theta^Tx_i  -\log(1 + \exp(\theta^T x_i)) ]$$

$$\frac{\partial}{\partial \theta_j}J(\theta) = -\frac{1}{n} \sum[y_i \frac{\partial}{\partial \theta_j} (\theta^Tx_i)  - \frac{\partial}{\partial \theta_j}\log(1 + \exp(\theta^T x_i)) ]$$

Knowing that:

$$\theta^Tx_i = \theta_1 {x_i}^{(1)} + \theta_2 {x_i}^{(2)} + \ldots + \theta_k {x_i}^{(k)}$$

$$\frac{\partial}{\partial \theta_j} (\theta^Tx_i) = x_i^{(j)}$$
$$\dfrac{\partial}{\partial \theta_j}\left(\log(1 + \exp(\theta^T x_i))\right) \underset{\log(u)^{'} = \dfrac{u^{'}}{u}} = \dfrac{\dfrac{\partial}{\partial \theta_j}(1 + \exp(\theta^T x_i))} {1 + \exp(\theta x_i)}$$

And:  

$$\dfrac{\partial}{\partial \theta_j}\left(\log(1 + \exp(\theta^T x_i))\right) \underset{\log(u)^{'} = \dfrac{u^{'}}{u}} = \dfrac{\dfrac{\partial}{\partial \theta_j}(1 + \exp(\theta^T x_i))} {1 + \exp(\theta x_i)}$$
$$\dfrac{\dfrac{\partial}{\partial \theta_j}(1 + \exp(\theta^T x_i))} {1 + \exp(\theta x_i)} = \dfrac{\dfrac{\partial}{\partial \theta_j}(\exp(\theta^T x_i))} {1 + \exp(\theta x_i)}$$
$$\dfrac{\dfrac{\partial}{\partial \theta_j}(\exp(\theta^T x_i))} {1 + \exp(\theta x_i)} \underset{\exp(u)^{'} = u^{'}\exp(u)} =  \dfrac{\dfrac{\partial}{\partial \theta_j}(\theta^T x_i) * (\exp(\theta^T x_i))} {1 + \exp(\theta x_i)}$$
$$\dfrac{\dfrac{\partial}{\partial \theta_j}(\theta^T x_i) * (\exp(\theta^T x_i))} {1 + \exp(\theta x_i)} = \frac{x_i^{(j)} * (\exp(\theta^T x_i))} {1 + \exp(\theta x_i)}$$
$$\dfrac{x_i^{(j)} * (\exp(\theta^T x_i))} {1 + \exp(\theta x_i)} = x_i^{(j)} * h_\theta(x_i)$$
$$\dfrac{\partial}{\partial \theta_j}J(\theta) = -\dfrac{1}{n} \sum[y_i x_i^{(j)}  - x_i^{(j)} h_\theta(x_i)]$$

$$-\dfrac{1}{n} \sum[y_i x_i^{(j)}  - x_i^{(j)} h_\theta(x_i)] = -\dfrac{1}{n} \sum[y_i - h_\theta(x_i) ] x_i^{(j)}$$

$$\frac{\partial}{\partial \theta_j}J(\theta) = \frac{1}{n} \sum[h_\theta(x_i) - y_i ] x_i^{(j)}$$

We know that the expression of the Gradient descent to update the weights is for the weight $\theta_j$ :  

$$\theta_j = \theta_j - \alpha \frac{\partial}{\partial \theta_j}J(\theta)$$

$$\theta_j = \theta_j - \frac{\alpha}{n} \sum[h_\theta(x_i) - y_i ] x_i^{(j)}$$

#### The Newton-Raphson algorithm

Others algorithms can be used to find the coefficients of the logistic regression. The Newton-Raphson algorithm is an alternative to the gradient descent.

The Newton-Raphson method uses both the first derivative (gradient) and the second derivative (Hessian) of the cost function. It approximates the cost function as a quadratic form and finds the minimum of this approximation.  

**Update rule**

$$\theta = \theta - H^{-1} \nabla J(\theta)$$

Logistic regression is a regression model used to predict the probability of a binary event based on one or more predictive variables. In logistic regression, the likelihood function is convex and can be maximized using the Newton-Raphson algorithm.  

The Newton-Raphson algorithm is an iterative method for finding the maximum of a function using its first and second derivatives. For logistic regression, the likelihood function is given by:

$L(\theta | X, y) = \prod(P(yi | x_i, \theta)^{yi} (1 - P(y_i | x_i, \theta))^{(1 - y_i)})$

where: 
* $\theta$ is the vector of logistic regression coefficients, 
* $X$ is the matrix of predictive variables, 
* $y$ is the vector of binary response variables, and
* $P(y_i | x_i, \theta)$ is the predicted probability of the binary event for observation $i$.

To maximize the likelihood function, the Newton-Raphson algorithm updates the coefficient vector $\theta$ at each iteration using the following formula:


$\theta_{i+1} = \theta_i - H^{-1} . g$  

where  

- $H = \dfrac{\partial^2L}{\partial \theta \partial\theta'}$ is the Hessian matrix of the likelihood function.  

- $g = \dfrac{\partial L}{\partial \theta }$ is the gradient vector of the likelihood function, and,  

- $\theta_i$ is the coefficient vector at iteration $i$. 

The Hessian matrix and gradient vector of the likelihood function are computed using the partial derivatives of the likelihood with respect to the coefficients $\theta$.  

Thus, in logistic regression, the Newton-Raphson algorithm is used to estimate the coefficients by maximizing the likelihood function. This allows the prediction of binary event probabilities based on the predictive variables.  

**Note**: The iterations stop when the difference between two successive solution vectors becomes negligible.

### Convexity

Convexity is a crucial property in optimization, as it ensures that any local minimum is also a global minimum. This makes it easier to find the optimal solution using methods like gradient descent.  

The Log Loss function is convex due to its logarithmic form, which is always convex for positive values. While convexity guarantees that a stationary point (where the derivative is zero) is a global minimum, it does not guarantee the existence of such a point.  

To prove the existence of a minimum, the function must also be bounded below and attain this lower bound. For Log Loss, the function is bounded below by zero (since the logarithm of a positive number is always defined) and reaches this bound when predictions are perfectly accurate (i.e., the predicted probability for the correct class is 1).  

Combining convexity with the fact that Log Loss is bounded below and attains its lower bound, we conclude that the function reaches a global minimum when predictions are perfectly correct.

## Coefficients significativity

The Wald statistic allows to test the coefficients significativity $\hat{w_j}$. Wald statistic is given by::    

$(\frac{\hat{w_j}}{\sigma(\hat{w_j})})^2$  

Under $H_0 : \{\hat{w_j} = 0 \} \Longrightarrow \frac{\hat{w_j}}{\sigma(\hat{w_j})} $ ~ $\mathcal{N}(0, 1)$

The added-value of the variable $X_j$ is only real if the Wald statistic > 4 $(3.84 = 1.96^2)$

$Wald > 4$    

$\iff (\frac{\hat{w_j}}{\sigma(\hat{w_j})})^2 > 4$  

$\iff \frac{\hat{w_j}}{\sigma(\hat{w_j})} > 2$  

$\iff \hat{w_j} > 2\sigma(\hat{w_j}) $  

$\iff \hat{w_j} - 2\sigma(\hat{w_j}) > 0$  

$\iff \hat{w_j}$ se trouve à plus de 2 écarts-type de 0  

$\iff $ l'intervalle de confiance de $\hat{w_j}$ ne contient pas 0 à 95%  

CQFD

## Model quality mesure (Deviance)

Cf. S.Tufféry p.315

$n:$ number of observations  
$k:$ number of features

$L(\omega_k)$ Likelihood of the "modèle ajusté"  

$L(\omega_0)$ Likelihood of the "modèle réduit à la constante"  

$L(\omega_{max})$ Likelihood of the "modèle saturé". The one the model will compare.  


The Deviance formula:  

$D(\omega_k) = -2[log(L(\omega_k)) - log(L(\omega_{max}))]$  $^{(*)}$

As the target is 0 or 1 $\Longrightarrow L(\omega_{max})=1 \Longrightarrow log(L(\omega_{max}))=0$  

$\Longrightarrow D(\omega_k) = -2[log(L(\omega_k))]$

(*) $D(\omega_k) = (\frac{log(L(\omega_k))}{log(L(\omega_{max}))}^2)$ 

The goal of the logistic regression is to maximise the Likelihood which is equivalent to minimize the Deviance.

The Deviance is equivalent to the SCE for the linear regression.